DOWNLOAD FOLDER

In [2]:
import os
import shutil
import kagglehub

test_dir = "benchmark_test_files"
os.makedirs(test_dir, exist_ok=True)

def copy_10_files(dataset_slug, extension, prefix):
    print(f" Downloading dataset: {dataset_slug}...")
    dataset_path = kagglehub.dataset_download(dataset_slug)
    
    copied = 0
    for root, dirs, files in os.walk(dataset_path):
        for file in files:
            if file.lower().endswith(extension) and copied < 10:
                source_path = os.path.join(root, file)
                dest_path = os.path.join(test_dir, f"{prefix}_{copied+1}{extension}")
                
                shutil.copy(source_path, dest_path)
                copied += 1
                
                if copied >= 10:
                    break
        if copied >= 10:
            break
            
    print(f"Pulling {copied} files {extension} from {dataset_slug}")


# 1. Text-Based PDFs 
copy_10_files("snehaanbhawal/resume-dataset", ".pdf", "text_based")

# 2. Image-Based PDFs 
copy_10_files("hadikp/resume-data-pdf", ".pdf", "image_based")

# 3. DOCX Files
copy_10_files("palaksood97/resume-dataset", ".docx", "word_based")

print(f"\n Folder '{test_dir}' is ready with {len(os.listdir(test_dir))} files.")

/home/jupyter-user/gender-classification-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pulling 10 files .pdf from snehaanbhawal/resume-dataset
Pulling 10 files .pdf from hadikp/resume-data-pdf


100%|██████████| 11.1M/11.1M [00:01<00:00, 6.87MB/s]

Extracting files...


Pulling 10 files .docx from palaksood97/resume-dataset

 Folder 'benchmark_test_files' is ready with 30 files.


FLOW

1. Check file extension
2. If .docx => use python-docx to return text
3. If .pdf => use PyMuPDF to extract text
4. If text length is greater than 0 => return text
5. If text length is 0 (meaning it's a scanned image) => trigger OCR Fallback (use easyOCR + PyMuPDF) => return text

In [7]:
pip install  python-docx docx2txt mammoth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [mammoth]m2/3 [mammoth]
Note: you may need to restart the kernel to use updated packages.


In [8]:
pip install pymupdf pdfplumber pypdf 

Note: you may need to restart the kernel to use updated packages.


In [20]:
pip install pdf2image easyocr paddleocr paddlepaddle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 MB 118.0 MB/s  0:00:010:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [paddlepaddle] [paddlepaddle]
Note: you may need to restart the kernel to use updated packages.


In [40]:
pip install rapidocr-onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 45.0 MB/s  0:00:00 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [21]:
import os
import time
import pandas as pd
import numpy as np
from PIL import Image

# --- Import Libraries ---
# 1. DOCX Libraries
import docx
import docx2txt
import mammoth

# 2. PDF Text Extraction Libraries
import pymupdf
import pdfplumber
import pypdf

In [51]:
# 3. OCR Engines 
print(" Initializing OCR Engines...")
try:
    import easyocr
    easy_reader = easyocr.Reader(['en'], gpu=False, verbose=False) # set gpu=True if CUDA is preferred
    EASY_READY = True
except Exception as e:
    print(f"EasyOCR Init Warning: {e}")
    EASY_READY = False
    
try:
    from paddleocr import PaddleOCR
    # FIX 1: Updated to use_textline_orientation
    # FIX 2: Removed show_log entirely
    # FIX 3: Kept enable_mkldnn=False to prevent the C++ crash
    paddle_reader = PaddleOCR(use_textline_orientation=False, lang='en', enable_mkldnn=False)
    PADDLE_READY = True
except Exception as e:
    print(f"PaddleOCR Init Warning: {e}")
    PADDLE_READY = False

try:
    from rapidocr_onnxruntime import RapidOCR
    rapid_reader = RapidOCR()
    RAPID_READY = True
except Exception as e:
    print(f"RapidOCR Init Warning: {e}")
    RAPID_READY = False

print(f"EasyOCR: {EASY_READY} | PaddleOCR: {PADDLE_READY} | RapidOCR: {RAPID_READY}")

 Initializing OCR Engines...


Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/jupyter-user/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/jupyter-user/.paddlex/official_models/UVDoc`.
Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/jupyter-user/.paddlex/official_models/PP-OCRv6_medium_det`.
Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/jupyter-user/.paddlex/official_models/PP-OCRv6_medium_rec`.


EasyOCR: True | PaddleOCR: True | RapidOCR: True


In [52]:
# --- Setup Directories ---
test_dir = "benchmark_test_files"
output_text_dir = "extracted_texts"
os.makedirs(output_text_dir, exist_ok=True) # Creates folder if it doesn't exist

# Helper function to save and overwrite text files
def save_text_to_file(original_file, library_name, extracted_text):
    if extracted_text:  # Only save if text was actually extracted
        safe_name = f"{original_file}_{library_name}.txt"
        out_path = os.path.join(output_text_dir, safe_name)
        # "w" mode guarantees the file is overwritten/updated on every run
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(extracted_text)

results = []

In [53]:
for file_name in sorted(os.listdir(test_dir)):
    file_path = os.path.join(test_dir, file_name)
    ext = file_name.lower().split('.')[-1]
    
    if ext not in ["pdf", "docx"]:
        continue
        
    print(f"Analyzing: {file_name}...")
    
    
    # A. DOCX COMPARISON
    if ext == "docx":
        # 1. python-docx
        t0 = time.time()
        try:
            doc = docx.Document(file_path)
            text_pdocx = "\n".join([p.text for p in doc.paragraphs])
            save_text_to_file(file_name, "python-docx", text_pdocx)
        except Exception: text_pdocx = ""
        t_pdocx = time.time() - t0
        results.append({"File": file_name, "Format": "DOCX", "Tool": "python-docx", "Method": "Native", "Time (s)": round(t_pdocx, 4), "Chars": len(text_pdocx)})
        
        # 2. docx2txt
        t0 = time.time()
        try:
            text_d2t = docx2txt.process(file_path)
            save_text_to_file(file_name, "docx2txt", text_d2t)
        except Exception: text_d2t = ""
        t_d2t = time.time() - t0
        results.append({"File": file_name, "Format": "DOCX", "Tool": "docx2txt", "Method": "Native", "Time (s)": round(t_d2t, 4), "Chars": len(text_d2t) if text_d2t else 0})
        
        # 3. mammoth
        t0 = time.time()
        try:
            with open(file_path, "rb") as docx_file:
                result_m = mammoth.extract_raw_text(docx_file)
                text_mammoth = result_m.value
                save_text_to_file(file_name, "mammoth", text_mammoth)
        except Exception: text_mammoth = ""
        t_mammoth = time.time() - t0
        results.append({"File": file_name, "Format": "DOCX", "Tool": "mammoth", "Method": "Native", "Time (s)": round(t_mammoth, 4), "Chars": len(text_mammoth)})

    # B. PDF COMPARISON (Digital)
    elif ext == "pdf":
        pdf_tools = {
            "PyMuPDF": lambda p: "".join([page.get_text() for page in pymupdf.open(p)]),
            "pdfplumber": lambda p: "".join([page.extract_text() or "" for page in pdfplumber.open(p).pages]),
            "pypdf": lambda p: "".join([page.extract_text() or "" for page in pypdf.PdfReader(p).pages])
        }
        
        file_needs_ocr = False
        
        for tool_name, extract_func in pdf_tools.items():
            t0 = time.time()
            try:
                extracted_text = extract_func(file_path).strip()
                save_text_to_file(file_name, tool_name, extracted_text)
            except Exception:
                extracted_text = ""
            t_exec = time.time() - t0
            
            # Check if digital extraction failed (< 100 characters implies a scanned image)
            if len(extracted_text) < 100:
                file_needs_ocr = True
                
            results.append({"File": file_name, "Format": "PDF", "Tool": tool_name, "Method": "Digital Text", "Time (s)": round(t_exec, 4), "Chars": len(extracted_text)})
            
        # C. OCR COMPARISON (Fallback)

        if file_needs_ocr:
            print(f"  -> {file_name} appears to be a scanned image. Running OCR engines...")
            
            doc = pymupdf.open(file_path)
            img_arrays = []
            for page in doc:
                pix = page.get_pixmap(dpi=150)
                img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
                img_arrays.append(np.array(img))
                
            # 1. EasyOCR
            if EASY_READY:
                t0 = time.time()
                try:
                    ocr_text_easy = ""
                    for img_arr in img_arrays:
                        res = easy_reader.readtext(img_arr, detail=0)
                        ocr_text_easy += " ".join(res) + "\n"
                    save_text_to_file(file_name, "EasyOCR", ocr_text_easy)
                except Exception: ocr_text_easy = ""
                t_easy = time.time() - t0
                results.append({"File": file_name, "Format": "PDF", "Tool": "EasyOCR", "Method": "OCR Fallback", "Time (s)": round(t_easy, 4), "Chars": len(ocr_text_easy.strip())})
                
            # 2. PaddleOCR
            if PADDLE_READY:
                t0 = time.time()
                try:
                    ocr_text_paddle = ""
                    for img_arr in img_arrays:
                        # 1. Save the array to a temporary physical file to bypass the C++ array bug
                        temp_img_path = "temp_paddle_img.jpg"
                        Image.fromarray(img_arr).save(temp_img_path)
                        
                        # 2. Use the new .predict() command to fix the warning, and pass the file path!
                        res = paddle_reader.predict(temp_img_path) 
                        
                        if res and res[0]:
                            line_texts = [line[1][0] for line in res[0]]
                            ocr_text_paddle += " ".join(line_texts) + "\n"
                            
                        # 3. Clean up the temporary file so we don't clutter your folder
                        if os.path.exists(temp_img_path):
                            os.remove(temp_img_path)
                            
                    save_text_to_file(file_name, "PaddleOCR", ocr_text_paddle)
                except Exception as e: 
                    print(f"  ❌ PaddleOCR crashed on {file_name}: {e}")
                    ocr_text_paddle = ""
                
                t_paddle = time.time() - t0
                results.append({"File": file_name, "Format": "PDF", "Tool": "PaddleOCR", "Method": "OCR Fallback", "Time (s)": round(t_paddle, 4), "Chars": len(ocr_text_paddle.strip())})
                
            # 3. RapidOCR
            if RAPID_READY:
                t0 = time.time()
                try:
                    ocr_text_rapid = ""
                    for img_arr in img_arrays:
                        res, _ = rapid_reader(img_arr)
                        if res:
                            # RapidOCR returns a list of tuples: (box, text, score)
                            line_texts = [line[1] for line in res]
                            ocr_text_rapid += " ".join(line_texts) + "\n"
                    save_text_to_file(file_name, "RapidOCR", ocr_text_rapid)
                except Exception: ocr_text_rapid = ""
                t_rapid = time.time() - t0
                results.append({"File": file_name, "Format": "PDF", "Tool": "RapidOCR", "Method": "OCR Fallback", "Time (s)": round(t_rapid, 4), "Chars": len(ocr_text_rapid.strip())})

Analyzing: image_based_1.pdf...
  -> image_based_1.pdf appears to be a scanned image. Running OCR engines...
Analyzing: image_based_10.pdf...
  -> image_based_10.pdf appears to be a scanned image. Running OCR engines...
Analyzing: image_based_2.pdf...
  -> image_based_2.pdf appears to be a scanned image. Running OCR engines...
Analyzing: image_based_3.pdf...
  -> image_based_3.pdf appears to be a scanned image. Running OCR engines...
Analyzing: image_based_4.pdf...
  -> image_based_4.pdf appears to be a scanned image. Running OCR engines...
Analyzing: image_based_5.pdf...
  -> image_based_5.pdf appears to be a scanned image. Running OCR engines...
Analyzing: image_based_6.pdf...
  -> image_based_6.pdf appears to be a scanned image. Running OCR engines...
Analyzing: image_based_7.pdf...
  -> image_based_7.pdf appears to be a scanned image. Running OCR engines...
Analyzing: image_based_8.pdf...
  -> image_based_8.pdf appears to be a scanned image. Running OCR engines...
Analyzing: image_

In [54]:
# --- Final Comparison ---

from IPython.display import HTML, display

df_benchmark = pd.DataFrame(results)

print("\n MULTI-LIBRARY BENCHMARK COMPLETE!")

df_display = df_benchmark.copy()
df_display.index = range(1, len(df_display) + 1)

display(HTML(df_display.to_html()))


 MULTI-LIBRARY BENCHMARK COMPLETE!


,File,Format,Tool,Method,Time (s),Chars
1,image_based_1.pdf,PDF,PyMuPDF,Digital Text,0.0016,0
2,image_based_1.pdf,PDF,pdfplumber,Digital Text,0.0022,0
3,image_based_1.pdf,PDF,pypdf,Digital Text,0.0013,0
4,image_based_1.pdf,PDF,EasyOCR,OCR Fallback,25.8758,3257
5,image_based_1.pdf,PDF,PaddleOCR,OCR Fallback,80.7493,29
6,image_based_1.pdf,PDF,RapidOCR,OCR Fallback,8.2612,2883
7,image_based_10.pdf,PDF,PyMuPDF,Digital Text,0.0026,0
8,image_based_10.pdf,PDF,pdfplumber,Digital Text,0.0021,0
9,image_based_10.pdf,PDF,pypdf,Digital Text,0.0012,0
10,image_based_10.pdf,PDF,EasyOCR,OCR Fallback,30.4769,2267


In [57]:
import pandas as pd
import numpy as np

# We use named aggregation to calculate both Time and Chars cleanly
latency_summary = df_benchmark.groupby(['Format', 'Method', 'Tool']).agg(
    Avg_Time=('Time (s)', 'mean'),
    Min_Time=('Time (s)', 'min'),
    Max_Time=('Time (s)', 'max'),
    Avg_Chars=('Chars', 'mean')
).reset_index()

# Rename columns for a cleaner presentation
latency_summary = latency_summary.rename(columns={
    'Avg_Time': 'Avg Time (s)', 
    'Min_Time': 'Min Time (s)', 
    'Max_Time': 'Max Time (s)',
    'Avg_Chars': 'Avg Chars'
})

# Round the average characters to whole numbers for a cleaner table
latency_summary['Avg Chars'] = latency_summary['Avg Chars'].round(0).astype(int)

# Sort so the fastest tool in each category is at the top
latency_summary = latency_summary.sort_values(by=['Format', 'Method', 'Avg Time (s)']).reset_index(drop=True)

print("AVERAGE LATENCY & CHARACTER SUMMARY TABLE")
print("=" * 70)
try:
    from IPython.display import display
    display(latency_summary)
except ImportError:
    print(latency_summary.to_string())

print("\n SPEED VS. YIELD DIFFERENTIALS")
print("=" * 70)

# Calculate exact speed multipliers for each category
categories = [
    ("DOCX", "Native"),
    ("PDF", "Digital Text"),
    ("PDF", "OCR Fallback")
]

for fmt, method in categories:
    subset = latency_summary[(latency_summary['Format'] == fmt) & (latency_summary['Method'] == method)]
    
    if subset.empty or len(subset) < 2:
        continue
        
    print(f"\n--- {fmt} ({method}) ---")
    fastest_row = subset.iloc[0]
    fastest_tool = fastest_row['Tool']
    fastest_time = fastest_row['Avg Time (s)']
    fastest_chars = fastest_row['Avg Chars']
    
    for _, row in subset.iloc[1:].iterrows():
        slower_tool = row['Tool']
        slower_time = row['Avg Time (s)']
        slower_chars = row['Avg Chars']
        
        # Prevent division by zero if a tool was absurdly fast (0.0s)
        if fastest_time > 0:
            multiplier = slower_time / fastest_time
            print(f" {fastest_tool} is {multiplier:.1f}x faster than {slower_tool} "
                  f"({fastest_time:.4f}s vs {slower_time:.4f}s) | 🔠 Chars: {fastest_chars} vs {slower_chars}")
        else:
            print(f" {fastest_tool} is instantly faster than {slower_tool} "
                  f"(0.000s vs {slower_time:.4f}s) | 🔠 Chars: {fastest_chars} vs {slower_chars}")

AVERAGE LATENCY & CHARACTER SUMMARY TABLE


,Format,Method,Tool,Avg Time (s),Min Time (s),Max Time (s),Avg Chars
0,DOCX,Native,docx2txt,0.026510,0.0084,0.0417,18543
1,DOCX,Native,python-docx,0.030660,0.0114,0.0482,15771
2,DOCX,Native,mammoth,0.489380,0.0423,1.1306,18470
3,PDF,Digital Text,PyMuPDF,0.007730,0.0013,0.0374,2809
4,PDF,Digital Text,pypdf,0.175315,0.0011,0.9500,2850
5,PDF,Digital Text,pdfplumber,0.260610,0.0017,1.2683,2806
6,PDF,OCR Fallback,RapidOCR,9.138580,5.9973,15.2732,2000
7,PDF,OCR Fallback,EasyOCR,37.291790,25.1036,66.5466,2231
8,PDF,OCR Fallback,PaddleOCR,76.182170,44.5247,152.9543,29



 SPEED VS. YIELD DIFFERENTIALS

--- DOCX (Native) ---
 docx2txt is 1.2x faster than python-docx (0.0265s vs 0.0307s) | 🔠 Chars: 18543 vs 15771
 docx2txt is 18.5x faster than mammoth (0.0265s vs 0.4894s) | 🔠 Chars: 18543 vs 18470

--- PDF (Digital Text) ---
 PyMuPDF is 22.7x faster than pypdf (0.0077s vs 0.1753s) | 🔠 Chars: 2809 vs 2850
 PyMuPDF is 33.7x faster than pdfplumber (0.0077s vs 0.2606s) | 🔠 Chars: 2809 vs 2806

--- PDF (OCR Fallback) ---
 RapidOCR is 4.1x faster than EasyOCR (9.1386s vs 37.2918s) | 🔠 Chars: 2000 vs 2231
 RapidOCR is 8.3x faster than PaddleOCR (9.1386s vs 76.1822s) | 🔠 Chars: 2000 vs 29


USING resume_examples

In [ ]:
# # --- Setup Directories ---
# test_dir = "resume_examples"
# output_text_dir = "extracted_texts_resume_ex"
# os.makedirs(output_text_dir, exist_ok=True) # Creates folder if it doesn't exist

# # Helper function to save and overwrite text files
# def save_text_to_file(original_file, library_name, extracted_text):
#     if extracted_text:  # Only save if text was actually extracted
#         safe_name = f"{original_file}_{library_name}.txt"
#         out_path = os.path.join(output_text_dir, safe_name)
#         # "w" mode guarantees the file is overwritten/updated on every run
#         with open(out_path, "w", encoding="utf-8") as f:
#             f.write(extracted_text)

# results = []

In [ ]:
# for file_name in sorted(os.listdir(test_dir)):
#     file_path = os.path.join(test_dir, file_name)
#     ext = file_name.lower().split('.')[-1]
    
#     if ext not in ["pdf", "docx"]:
#         continue
        
#     print(f"Analyzing: {file_name}...")
    
    
#     # A. DOCX COMPARISON
#     if ext == "docx":
#         # 1. python-docx
#         t0 = time.time()
#         try:
#             doc = docx.Document(file_path)
#             text_pdocx = "\n".join([p.text for p in doc.paragraphs])
#             save_text_to_file(file_name, "python-docx", text_pdocx)
#         except Exception: text_pdocx = ""
#         t_pdocx = time.time() - t0
#         results.append({"File": file_name, "Format": "DOCX", "Tool": "python-docx", "Method": "Native", "Time (s)": round(t_pdocx, 4), "Chars": len(text_pdocx)})
        
#         # 2. docx2txt
#         t0 = time.time()
#         try:
#             text_d2t = docx2txt.process(file_path)
#             save_text_to_file(file_name, "docx2txt", text_d2t)
#         except Exception: text_d2t = ""
#         t_d2t = time.time() - t0
#         results.append({"File": file_name, "Format": "DOCX", "Tool": "docx2txt", "Method": "Native", "Time (s)": round(t_d2t, 4), "Chars": len(text_d2t) if text_d2t else 0})
        
#         # 3. mammoth
#         t0 = time.time()
#         try:
#             with open(file_path, "rb") as docx_file:
#                 result_m = mammoth.extract_raw_text(docx_file)
#                 text_mammoth = result_m.value
#                 save_text_to_file(file_name, "mammoth", text_mammoth)
#         except Exception: text_mammoth = ""
#         t_mammoth = time.time() - t0
#         results.append({"File": file_name, "Format": "DOCX", "Tool": "mammoth", "Method": "Native", "Time (s)": round(t_mammoth, 4), "Chars": len(text_mammoth)})

#     # B. PDF COMPARISON (Digital)
#     elif ext == "pdf":
#         pdf_tools = {
#             "PyMuPDF": lambda p: "".join([page.get_text() for page in pymupdf.open(p)]),
#             "pdfplumber": lambda p: "".join([page.extract_text() or "" for page in pdfplumber.open(p).pages]),
#             "pypdf": lambda p: "".join([page.extract_text() or "" for page in pypdf.PdfReader(p).pages])
#         }
        
#         file_needs_ocr = False
        
#         for tool_name, extract_func in pdf_tools.items():
#             t0 = time.time()
#             try:
#                 extracted_text = extract_func(file_path).strip()
#                 save_text_to_file(file_name, tool_name, extracted_text)
#             except Exception:
#                 extracted_text = ""
#             t_exec = time.time() - t0
            
#             # Check if digital extraction failed (< 100 characters implies a scanned image)
#             if len(extracted_text) < 100:
#                 file_needs_ocr = True
                
#             results.append({"File": file_name, "Format": "PDF", "Tool": tool_name, "Method": "Digital Text", "Time (s)": round(t_exec, 4), "Chars": len(extracted_text)})
            
#         # C. OCR COMPARISON (Fallback)

#         if file_needs_ocr:
#             print(f"  -> {file_name} appears to be a scanned image. Running OCR engines...")
            
#             doc = pymupdf.open(file_path)
#             img_arrays = []
#             for page in doc:
#                 pix = page.get_pixmap(dpi=150)
#                 img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
#                 img_arrays.append(np.array(img))
                
#             # 1. EasyOCR
#             if EASY_READY:
#                 t0 = time.time()
#                 try:
#                     ocr_text_easy = ""
#                     for img_arr in img_arrays:
#                         res = easy_reader.readtext(img_arr, detail=0)
#                         ocr_text_easy += " ".join(res) + "\n"
#                     save_text_to_file(file_name, "EasyOCR", ocr_text_easy)
#                 except Exception: ocr_text_easy = ""
#                 t_easy = time.time() - t0
#                 results.append({"File": file_name, "Format": "PDF", "Tool": "EasyOCR", "Method": "OCR Fallback", "Time (s)": round(t_easy, 4), "Chars": len(ocr_text_easy.strip())})
                
#             # 2. PaddleOCR
#             if PADDLE_READY:
#                 t0 = time.time()
#                 try:
#                     ocr_text_paddle = ""
#                     for img_arr in img_arrays:
#                         # 1. Save the array to a temporary physical file to bypass the C++ array bug
#                         temp_img_path = "temp_paddle_img.jpg"
#                         Image.fromarray(img_arr).save(temp_img_path)
                        
#                         # 2. Use the new .predict() command to fix the warning, and pass the file path!
#                         res = paddle_reader.predict(temp_img_path) 
                        
#                         if res and res[0]:
#                             line_texts = [line[1][0] for line in res[0]]
#                             ocr_text_paddle += " ".join(line_texts) + "\n"
                            
#                         # 3. Clean up the temporary file so we don't clutter your folder
#                         if os.path.exists(temp_img_path):
#                             os.remove(temp_img_path)
                            
#                     save_text_to_file(file_name, "PaddleOCR", ocr_text_paddle)
#                 except Exception as e: 
#                     print(f"  ❌ PaddleOCR crashed on {file_name}: {e}")
#                     ocr_text_paddle = ""
                
#                 t_paddle = time.time() - t0
#                 results.append({"File": file_name, "Format": "PDF", "Tool": "PaddleOCR", "Method": "OCR Fallback", "Time (s)": round(t_paddle, 4), "Chars": len(ocr_text_paddle.strip())})
                
#             # 3. RapidOCR
#             if RAPID_READY:
#                 t0 = time.time()
#                 try:
#                     ocr_text_rapid = ""
#                     for img_arr in img_arrays:
#                         res, _ = rapid_reader(img_arr)
#                         if res:
#                             # RapidOCR returns a list of tuples: (box, text, score)
#                             line_texts = [line[1] for line in res]
#                             ocr_text_rapid += " ".join(line_texts) + "\n"
#                     save_text_to_file(file_name, "RapidOCR", ocr_text_rapid)
#                 except Exception: ocr_text_rapid = ""
#                 t_rapid = time.time() - t0
#                 results.append({"File": file_name, "Format": "PDF", "Tool": "RapidOCR", "Method": "OCR Fallback", "Time (s)": round(t_rapid, 4), "Chars": len(ocr_text_rapid.strip())})

In [ ]:
# # --- Final Comparison ---

# from IPython.display import HTML, display

# df_benchmark = pd.DataFrame(results)

# print("\n MULTI-LIBRARY BENCHMARK COMPLETE!")

# df_display = df_benchmark.copy()
# df_display.index = range(1, len(df_display) + 1)

# display(HTML(df_display.to_html()))

In [ ]:
# import pandas as pd
# import numpy as np

# # We use named aggregation to calculate both Time and Chars cleanly
# latency_summary = df_benchmark.groupby(['Format', 'Method', 'Tool']).agg(
#     Avg_Time=('Time (s)', 'mean'),
#     Min_Time=('Time (s)', 'min'),
#     Max_Time=('Time (s)', 'max'),
#     Avg_Chars=('Chars', 'mean')
# ).reset_index()

# # Rename columns for a cleaner presentation
# latency_summary = latency_summary.rename(columns={
#     'Avg_Time': 'Avg Time (s)', 
#     'Min_Time': 'Min Time (s)', 
#     'Max_Time': 'Max Time (s)',
#     'Avg_Chars': 'Avg Chars'
# })

# # Round the average characters to whole numbers for a cleaner table
# latency_summary['Avg Chars'] = latency_summary['Avg Chars'].round(0).astype(int)

# # Sort so the fastest tool in each category is at the top
# latency_summary = latency_summary.sort_values(by=['Format', 'Method', 'Avg Time (s)']).reset_index(drop=True)

# print("AVERAGE LATENCY & CHARACTER SUMMARY TABLE")
# print("=" * 70)
# try:
#     from IPython.display import display
#     display(latency_summary)
# except ImportError:
#     print(latency_summary.to_string())

# print("\n SPEED VS. YIELD DIFFERENTIALS")
# print("=" * 70)

# # Calculate exact speed multipliers for each category
# categories = [
#     ("DOCX", "Native"),
#     ("PDF", "Digital Text"),
#     ("PDF", "OCR Fallback")
# ]

# for fmt, method in categories:
#     subset = latency_summary[(latency_summary['Format'] == fmt) & (latency_summary['Method'] == method)]
    
#     if subset.empty or len(subset) < 2:
#         continue
        
#     print(f"\n--- {fmt} ({method}) ---")
#     fastest_row = subset.iloc[0]
#     fastest_tool = fastest_row['Tool']
#     fastest_time = fastest_row['Avg Time (s)']
#     fastest_chars = fastest_row['Avg Chars']
    
#     for _, row in subset.iloc[1:].iterrows():
#         slower_tool = row['Tool']
#         slower_time = row['Avg Time (s)']
#         slower_chars = row['Avg Chars']
        
#         # Prevent division by zero if a tool was absurdly fast (0.0s)
#         if fastest_time > 0:
#             multiplier = slower_time / fastest_time
#             print(f" {fastest_tool} is {multiplier:.1f}x faster than {slower_tool} "
#                   f"({fastest_time:.4f}s vs {slower_time:.4f}s) | 🔠 Chars: {fastest_chars} vs {slower_chars}")
#         else:
#             print(f" {fastest_tool} is instantly faster than {slower_tool} "
#                   f"(0.000s vs {slower_time:.4f}s) | 🔠 Chars: {fastest_chars} vs {slower_chars}")